# Aim
<div style = "text-align: justify">Aim is mitigate data imbalance using <b>Undersampling, Oversampling, and WCGAN with modified Loss function and Spectral Normalization.</b> It is observed that our model performs better than the other two conventional techniques.</div>

# Dataset
<div style = "text-align: justify">The dataset was taken from Pozzolo et al. (2015). The dataset can be found <a href = "https://www.kaggle.com/mlg-ulb/creditcardfraud"><b>here.</b></a> The dataset consists of credit card transactions made by European cardholders in September, 2013. It has <b>492 fraudulent samples</b> out of 284807 total transactions, which makes upto <b>0.172%</b> of the entire dataset. The imbalance is clearly visible.</div>

# Structure
<div style = "text-align: justify">The dataset has 31 columns, which includes <b>Time</b> elapsed between each transaction and the first transaction in the dataset, <b>Amount</b> of transaction and <b>28 hidden features</b> (due to confidentiality issues). Class <b>1</b> represent fraud samples and class <b>0</b> represents normal samples.</div>

In [ ]:
import numpy as np
import pandas
import seaborn
import tensorflow as tf
import matplotlib.pyplot as plt
from tabulate import tabulate
from tensorflow import keras

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pandas.read_csv('../input/creditcardfraud/creditcard.csv')

In [ ]:
df.head()

In [ ]:
print('Number of missing values in dataset : ' + str(df.isna().sum().sum()))
print('Number of Fraudulent samples in  df : ' + str(df[df['Class'] == 1].shape[0]))
print('Number of Normal transactions in df : ' + str(df[df['Class'] == 0].shape[0]))

In [ ]:
plt.figure(figsize = (15,15))
mat = df.corr()
seaborn.heatmap(mat, vmin = -1.0, square = True)

<div style = "text-align: justify">No strong correlation was observed. So, all the features will be used for training the model. Now we should normalize the dataset to have <b>zero mean and unit deviation</b>. Then scale the dataframe between -1 and +1.</div>

# Normalize the features

In [ ]:
X = np.array(df.drop(['Class'], axis = 1))
y = np.array(df['Class'])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
X = MinMaxScaler(feature_range = (-1,1)).fit_transform(X)

In [ ]:
info=[[df.columns[j],X[:,j].shape[0],X[:,j].min(),X[:,j].max(), X[:,j].mean(), X[:,j].std()] for j in range(X.shape[1])]
print(tabulate(info, headers = ['Column', 'Count', 'Minimum', 'Maximum', 'Mean', 'Standard dev'], tablefmt =  'orgtbl'))

# train-test divide

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.2, random_state = 1)

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
print(X_valid.shape)
print(y_valid.shape)

In [ ]:
print(f'Fraudulent samples in train set : {y_train[y_train == 1].shape[0]}')
print(f'Fraudulent samples in valid set : {y_valid[y_valid == 1].shape[0]}')

In [ ]:
info = [
    'normal',  # 0
    'fraud' ,  # 1
]

# Undersampling
<div style = "text-align: justify">Undersampling involves downsampling the majority samples, to reduce imbalance in the dataset. The problem with this approach is the <b>information loss</b> it causes. The classifier model won't get to see all possible samples and this will affect its performance. <b>We will use RandomUnderSampler() method from imblearn package.</b></div>

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
X_under, y_under = RandomUnderSampler(sampling_strategy = 1.00, random_state = 1).fit_resample(X_train, y_train)

In [ ]:
print(X_under.shape)
print(y_under.shape)

In [ ]:
print(X_valid.shape)
print(y_valid.shape)

In [ ]:
value = []
for x in y_under :
    value.append(info[x])

In [ ]:
seaborn.histplot(pandas.DataFrame({'id' : value}), x = 'id')

### fit for the RandomForest model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model = RandomForestClassifier(criterion = 'gini',
                               max_features = None,
                               min_samples_split = 2,
                               random_state = 1, verbose = 1)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold

In [ ]:
search = RandomizedSearchCV(estimator = model, verbose = 1,
                            param_distributions = {'n_estimators' : [100,110, 135]},
                            cv = KFold(n_splits = 2, shuffle=True, random_state=1))

In [ ]:
search.fit(X_under, y_under)

In [ ]:
print(search.best_params_)

In [ ]:
model.n_estimators = search.best_params_['n_estimators']

In [ ]:
model.fit(X_under, y_under)

In [ ]:
y_pred = model.predict(X_valid)

### evaluate on testing data

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
print(classification_report(y_valid, y_pred, target_names = info))

In [ ]:
cm = confusion_matrix(y_valid, y_pred)

In [ ]:
plt.figure(figsize = (8,8))
ax = seaborn.heatmap(cm, cmap=plt.cm.Greens, annot=True, square=True,
                     xticklabels = info,
                     yticklabels = info)

ax.set_ylabel('GroundTruth',fontsize=20)
ax.set_xlabel('Predictions',fontsize=20)

<div style = "text-align: justify">It may look that the model is doing not very bad, <b>but it is !!</b> The model fauled for fraud samples, which were the real target to be classified. <b>Notice that out of 9 of the fraud transactions were labeled as normal.</b> This can't be good for any financial purposes.</div>

# Oversampling
<div style = "text-align: justify">Oversampling involves creating samples for the minority class. There are sveral ways to do this. <b>RandomOversampling</b> copies a minority sample and adds it back to the original set. The model however, does not learn anything new from the sampled examples. <b>SMOTE (or Sythetic Minority Oversampling Technique)</b> selects a minority sample and one of its k nearest-neighbours. Then <b>a line is drawn between the two ponts and a new sample is generated anywhere on this line in the feature space.</b></div>

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
X_over, y_over = SMOTE(sampling_strategy = 1.00, random_state = 1).fit_resample(X_train, y_train)

In [ ]:
'''
Had todecrease the size,
for faster computation.I
will use this much data.
'''
X_over, _, y_over, _ = train_test_split(X_over, y_over, test_size = 0.8, random_state = 1)

In [ ]:
print(X_over.shape)
print(y_over.shape)

In [ ]:
print(X_valid.shape)
print(y_valid.shape)

In [ ]:
value = []
for x in y_over :
    value.append(info[x])

In [ ]:
seaborn.histplot(pandas.DataFrame({'id' : value}), x = 'id')

### fit the RandomForest model

In [ ]:
model = RandomForestClassifier(criterion = 'gini',
                               max_features = None,
                               min_samples_split = 2,
                               random_state = 1, verbose = 1)

In [ ]:
search = RandomizedSearchCV(estimator = model, verbose = 5,
                            param_distributions = {'n_estimators' : [135,150, 175]},
                            cv = KFold(n_splits = 2, shuffle=True, random_state=1))

In [ ]:
search.fit(X_over, y_over)

In [ ]:
print(search.best_params_)

In [ ]:
model.n_estimators = search.best_params_['n_estimators']

In [ ]:
model.fit(X_over, y_over)

In [ ]:
y_pred = model.predict(X_valid)

### Evaluate performance on testing data

In [ ]:
print(classification_report(y_valid, y_pred, target_names = info))

In [ ]:
cm = confusion_matrix(y_valid, y_pred)

In [ ]:
plt.figure(figsize = (8,8))
ax = seaborn.heatmap(cm, cmap=plt.cm.Blues, annot=True, square=True,
                     xticklabels = info,
                     yticklabels = info)

ax.set_ylabel('GroundTruth',fontsize=20)
ax.set_xlabel('Predictions',fontsize=20)

<div style = "text-align: justify">Again as stated earlier, the performance of model is not good. <b>It fails to flag a significant number of fraud samples.</b> Also, many normal transactions get stuck as frauds.</div>

# Distribution of means
<div style = "text-align: justify">To see how different the features of the <b>fraud samples are from the normal samples</b>, we will plot the means of both the classes for each feature.</div>

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
seaborn.set_theme(style = "darkgrid")

In [ ]:
plt.figure(figsize = (20,70))
i = 0
while i < 30 :
    
    plt.subplot(10, 3, i + 1)
    
    FRAUD = np.mean(X_train[y_train == 1][:,i])
    LEGIT = np.mean(X_train[y_train == 0][:,i])
    temp  = pandas.DataFrame({'class' : ['Fraud', 'Legit'], 'means' : [FRAUD, LEGIT]})
    
    seaborn.pointplot(data  = temp,x = 'class',y = 'means')
    plt.title(f'mean dist. {df.columns[i]}', fontsize = 20)
    i += 1
plt.show()

# WGAN with Weight Clipping

### Problem
<div style = "text-align: justify">The vanilla GAN models use <b>J-S Divergence</b> as the discriminator's cost, and had unstable training due to <b>exploding and vanishing gradients.</b> Due to this GANs had the possibility of suffering from mode collapse.</div>

### Solution
<div style = "text-align: justify">Solution to the issue was <b>Wasserstein distance.</b> It can be seen as the minimum cost path to convert generated distribution to real distribution. The cost function has smoother gradients. The equation,</div>
</br>

![func](https://miro.medium.com/max/3600/1*6y-tz57odJpHh4pwRfXACw.png)

</br>
<div style = "text-align: justify">Here, f is the 1-Lipchitz function meaning the norm of its gradients is always less than or equal to 1. <b>The 1-Lipchitz constrain is implemented by clipping the weights of the discriminator.</b></div>

In [ ]:
import cv2
from tqdm import tqdm

In [ ]:
img = cv2.imread('../input/cganimage/1*CLgoDPChiyvl7dToEwkWGw.png')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.figure(figsize = (20,20))
plt.imshow(img)
plt.axis('off')

### Conditional part
<div style = "text-align: justify">For targeted generation of feature space based on the clss label, label will be added to both the Discriminator and the Generator. <b>The Discriminator judges if the feature vector is the correct representation of the label, while the Generator tries to create feature vector depending on the label.</b></div>

</br>

![](https://miro.medium.com/max/1050/1*l2tSqFN0Afwizm4LgalCGg.png)

In [ ]:
from keras.initializers import RandomNormal
from keras.layers import Activation
from keras.layers import Dropout
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import LeakyReLU
from keras.layers import Concatenate
from keras.layers import BatchNormalization

In [ ]:
init = RandomNormal(mean = 0.0, stddev = 0.02)

**Hyperparameter set**

In [ ]:
BATCH  = 64
BUFFER = 400
W_CLIP = 0.01
LABELS_DIM = 1
ALPHA = 0.00005
LATENT_SPACE = (100,)
LABELS_SPACE = (1,)
LATENT_DIM = 100
N_CRITIC = 5
N_GEN = 1

In [ ]:
'''
Combination of Dense, Batch
Norm and leakyRELU with 0.2
'''

def dens_batch_relu (hiddenx, x, batchnorm) :
    
    y = Dense(hiddenx, use_bias = False, kernel_initializer =  init)(x)
    y = BatchNormalization()(y)
    y = LeakyReLU(alpha=0.2)(y)
    
    return y # the output tensor

In [ ]:
'''
Generator model
'''
def generator_model (hidden1) :
    
    inp = Input(LATENT_SPACE)
    lab = Input(LABELS_SPACE)
    con = Concatenate()([inp, lab])
    h00 = dens_batch_relu(hiddenx = hidden1, x = con, batchnorm = True)
    out = Dense(30, activation = 'tanh',kernel_initializer = init)(h00)
    
    return Model(inputs = [inp,lab], outputs = out, name = 'Generator')

'''
WC_critic model
'''
def wc_critic_model (hidden1) :
    
    inp = Input(shape =(30,))
    lab = Input(LABELS_SPACE)
    con = Concatenate()([inp, lab])
    h00 = dens_batch_relu(hiddenx = hidden1, x = con, batchnorm = True)
    out = Dense(1,activation = 'linear',kernel_initializer = init)(h00)
    
    return Model(inputs = [inp,lab], outputs = out, name = 'WC_critic')

In [ ]:
generator = generator_model(64)
wc_critic = wc_critic_model(32)

In [ ]:
from keras.utils import plot_model

In [ ]:
plot_model(generator, './generator.png', show_shapes = True)

In [ ]:
plot_model(wc_critic, './WC_critic.png', show_shapes = True)

### Loss function

In [ ]:
import tensorflow as tf

In [ ]:
def wc_critic_loss (critic_fake, critic_real) :
    return  tf.reduce_mean(critic_fake) - tf.reduce_mean(critic_real)

def generator_loss (critic_fake) :
    return -tf.reduce_mean(critic_fake)

### Optimizers

In [ ]:
wc_critic_optimizer = tf.keras.optimizers.RMSprop(learning_rate = ALPHA)
generator_optimizer = tf.keras.optimizers.RMSprop(learning_rate = ALPHA)

### tensorflow datasets

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train , y_train))
train_dataset = train_dataset.shuffle(BUFFER).batch(batch_size = BATCH)

### fit() function

In [ ]:
print(generator.summary())
print(wc_critic.summary())

In [ ]:
@tf.function
def train_on_batch (real_image, label) :
    
    '''
    Critic
    '''
    i = 0
    while i < N_CRITIC :
        with tf.GradientTape(persistent=True, watch_accessed_variables = True) as tape :
            noise = tf.random.normal(shape = (BATCH, LATENT_DIM))
            fake_image = generator([noise ,label], training=True)
            opfake = wc_critic([fake_image,label], training=True)
            opreal = wc_critic([real_image,label], training=True)
            
            loss = wc_critic_loss(critic_fake = opfake, critic_real = opreal)
        
        grads = tape.gradient(loss,wc_critic.trainable_variables)
        wc_critic_optimizer.apply_gradients(zip(grads, wc_critic.trainable_variables))
        
        # ====================
        #     Weight clip
        # ====================
        for x in range(len(wc_critic.trainable_variables)) :
            wc_critic.trainable_variables[x].assign(tf.clip_by_value(wc_critic.trainable_variables[x], -W_CLIP, W_CLIP))
        i += 1
    
    '''
    Generator
    '''
    j = 0
    while j < N_GEN :
        with tf.GradientTape(persistent=True, watch_accessed_variables = True) as tape :
            noise = tf.random.normal(shape = (BATCH, LATENT_DIM))
            fake_image = generator([noise ,label], training=True)
            opfake = wc_critic([fake_image,label], training=True)
            
            loss = generator_loss(critic_fake = opfake)
        
        grads = tape.gradient(loss,generator.trainable_variables)
        generator_optimizer.apply_gradients(zip(grads, generator.trainable_variables))
        j += 1

In [ ]:
logs = {
    'WC_critic_loss' : [],
    'generator_loss' : [],
}

In [ ]:
def fit (EPOCHS = 100) :
    
    print('WC_Critic')
    
    for epoch in range(EPOCHS) :
        
        # =======================
        #      Model Training
        # =======================
        print(f'{epoch} out of {EPOCHS}')
        for n, (real_image, label) in train_dataset.enumerate() :
            if n ==  3560 :
                print('#....End')
                break
            if n%100 == 0 :
                print('#',end='')
            train_on_batch(real_image,label)
        
        # =======================
        #      Log the losses
        # =======================
        for real_image, label in train_dataset.take(1) :
            
            noise = tf.random.normal(shape = (BATCH , LATENT_DIM))
            fake_image = generator([noise ,label], training=False)
            opfake = wc_critic([fake_image,label], training=False)
            opreal = wc_critic([real_image,label], training=False)
            
            loss1 = wc_critic_loss(critic_fake = opfake, critic_real = opreal)
            loss2 = generator_loss(critic_fake = opfake)
            logs['WC_critic_loss'].append(loss1.numpy())
            logs['generator_loss'].append(loss2.numpy())
            
            
            if epoch%50 == 0 :
                
                # ===========================
                #    Plot the distribution
                # ===========================
                plt.figure(figsize = (20,70))
                
                i = 0
                while i < 30 :
                    
                    plt.subplot(10, 3, i+1)
                    seaborn.kdeplot(real_image[:,i], color = 'r')
                    seaborn.kdeplot(fake_image[:,i], color = 'b')
                    plt.legend(['Real image', 'generated image'])
                    plt.title(f'{df.columns[i]}', fontsize = 20 )
                    
                    i += 1
                plt.show()
                
                # ===========================
                #     Plot the loss func
                # ===========================
                plt.figure(figsize = (20,5))
                plt.plot(logs['WC_critic_loss'])
                plt.plot(logs['generator_loss'])
                plt.legend(['WC_critic', 'Generator'])
                plt.title('Loss in terms Epochs', fontsize = 20 )
                plt.xlabel('Epochs')
                plt.ylabel('Losses')
                plt.show()
                
                # ===========================
                #       Save the models
                # ===========================
                print('Saving the models....')
                generator.save(f'./generator_{epoch}_epochs.h5')
                wc_critic.save(f'./wc_critic_{epoch}_epochs.h5')

In [ ]:
fit(EPOCHS = 1001)

# WGAN with Spectral Normalization
<div style = "text-align: justify">The condition for <b>critic being 1-Lipchitz</b> can also be applied by using Spectral normalization, instead of weight clipping. The formula for norm is,</div>

![Spec norm](https://miro.medium.com/max/3600/1*5AfPcYEv29KcJ9cMnoGU7w.jpeg)

<div style = "text-align: justify">The spectral norm of an array A is the maximum sigular value of matrix A. The singular value of A can be calculated using <b>Singular Value Decomposition (or SVD)</b>.</div>

</br>

![img](https://miro.medium.com/max/3600/0*4_rhGcIcvHV1fFy1.jpeg)

</br>
<div style = "text-align: justify">Here, U and V are the eigen vectors for AA' and A'A, repectively, and S contains the square root of these eigen values. Both AA' and A'A have the same positive eigen values. Ultimately, the weights of each layer are divided by their norms to <b>make each layer obey the 1-Lipchitz constraint.</b></div>

In [ ]:
def dens_batch_norm (x, hidden_nodes) :
    
    y = SpectralNormalization(Dense(hidden_nodes,
                                    use_bias = False,
                                    kernel_initializer = init,
                                    kernel_regularizer = keras.regularizers.l2()))(x)
    
    y = BatchNormalization()(y)
    y = Dropout(.2)(y)
    y = LeakyReLU()(y)
    
    return y # output

'''
Critic using Spectral
Norm to enforce 1-lip
-chitz condition.
'''
def SN_critic_model (hidden0, hidden1) :
    
    inp = Input(shape = (30,))
    h00 = dens_batch_norm(x = inp, hidden_nodes = hidden0)
    h01 = dens_batch_norm(x = h00, hidden_nodes = hidden1)
    out = Dense(1,'linear',kernel_initializer = init)(h01)
    
    return Model(inputs = inp, outputs = out, name = 'SN')

In [ ]:
SN_critic = SN_critic_model(64, 5)
generator = generator_model(64)

In [ ]:
plot_model(SN_critic, './SN_critic.png', show_shapes = True)

In [ ]:
print(generator.summary())
print(SN_critic.summary())

In [ ]:
def SN_critic_loss (critic_gen_output, critic_tar_output) :
    return -(tf.reduce_mean(critic_tar_output) - tf.reduce_mean(critic_gen_output))

In [ ]:
SN_critic_optimizer = keras.optimizers.RMSprop(learning_rate = ALPHA)
generator_optimizer = keras.optimizers.RMSprop(learning_rate = ALPHA)

In [ ]:
@tf.function
def train_on_batch (tar_images) :
    
    '''
    Train Critic for
    C_ITER times.
    '''
    
    for _ in range(C_ITER) :
        
        with tf.GradientTape(persistent = True) as tape :
            
            '''
            Output
            '''
            gen_images = generator(tf.random.normal((BATCH,LATENT_SPACE)),training=True)
            critic_gen_output = SN_critic(gen_images, training = True)
            critic_tar_output = SN_critic(tar_images, training = True)
            
            '''
            Losses
            '''
            loss = SN_critic_loss(critic_gen_output,critic_tar_output)
        
        SN_grad = tape.gradient(loss  , SN_critic.trainable_variables)
        SN_critic_optimizer.apply_gradients(zip(SN_grad, SN_critic.trainable_variables))
    
    '''
    Train  Generator
    for G_ITER times.
    '''
    for _ in range(G_ITER) :
        
        with tf.GradientTape(persistent = True) as tape :
            
            '''
            Output
            '''
            gen_images = generator(tf.random.normal((BATCH,LATENT_SPACE)),training=True)
            critic_gen_output = SN_critic(gen_images, training = True)
            
            '''
            Losses
            '''
            loss = generator_loss(critic_gen_output=critic_gen_output)
        
        GE_grad = tape.gradient(loss  , generator.trainable_variables)
        generator_optimizer.apply_gradients(zip(GE_grad, generator.trainable_variables))

### fit()

In [ ]:
logs = {
    'SN_critic_loss' : [],
    'generator_loss' : [],
}

def fit (EPOCHS = 2500) :
    
    print('CriticwithSpecNorm')
    print(f'Epochs : {EPOCHS}')
    
    for epoch  in range(EPOCHS) :
        
        print(f'{epoch} out of {EPOCHS}')
        
        for n, tar_images in train_dataset.enumerate() :
            
            if n ==  3560 :
                print('#....End')
            if n%100 == 0 :
                print('#',end='')
            train_on_batch(tar_images)
        
        '''
        Compute and Log
        the losses.
        '''
        for tar_images in train_dataset.take(1) :
            
            '''
            Output
            '''
            gen_images = generator(tf.random.normal((BATCH, LATENT_SPACE)), training  =  True)
            critic_gen_output = SN_critic(gen_images, training = True)
            critic_tar_output = SN_critic(tar_images, training = True)
            
            '''
            Losses
            '''
            logs['SN_critic_loss'].append(SN_critic_loss(critic_gen_output,critic_tar_output))
            logs['generator_loss'].append(generator_loss(critic_gen_output))

# Plot the loss function

In [ ]:
fit(EPOCHS = 200)

In [ ]:
plt.figure(figsize = (20,5))
plt.plot(logs['SN_critic_loss'])
plt.plot(logs['generator_loss'])
plt.legend(['SN_critic', 'gen'])
plt.xlabel('Epochs')
plt.ylabel('Losses')
plt.title('Losses wrt. Epochs (Spectral Norm)', fontsize = 15)

In [ ]:
generator.save('./generator_with_spec_normal.h5')